# Lab 3 demo: NorthStar retention offers (Track B, RAG)

End-to-end run of the offer pipeline: policy chunks → Titan embeddings → FAISS retrieval → Claude Haiku 4.5 offer, then the RAGAS results from `docs/lab3-evidence/rag_eval.csv`.

Run with **Kernel → Restart & Run All** from the repo (needs AWS credentials with Bedrock access).

In [1]:
import os, sys, json
# the notebook lives in notebooks/; run everything from the repo root
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.insert(0, os.getcwd())
print('repo root:', os.getcwd())

repo root: /Users/santiago/Desktop/401/northstar/cs401r-gbolano


## 1. The index

`rag/ingest.py` split the 4 policy docs into section-level chunks, prefixed each with its document title and ID, embedded them with Titan Text Embeddings V2 (1024-d, normalized) and stored them in a FAISS inner-product index (= cosine similarity).

In [2]:
import faiss
chunks = json.load(open('rag/index/chunks.json'))
index = faiss.read_index('rag/index/policies.faiss')
print(f'{len(chunks)} chunks, {index.ntotal} vectors, dim {index.d}')
for c in chunks[:5]:
    print(c['id'], '|', next(l for l in c['text'].split('\n') if l.startswith('## '))[:70])

29 chunks, 29 vectors, dim 1024
KB-FAQ-022-1 | ## Returns
KB-FAQ-022-2 | ## Loyalty
KB-FAQ-022-3 | ## Shipping
KB-FAQ-022-4 | ## Orders and Account
KB-FAQ-022-5 | ## Guidance for Automated Agents


## 2. Retrieval: single query vs multi-query

Baseline used one tier query, so every customer got almost the same chunks. Multi-query asks one question per need (tier benefits / returns+shipping for how they shop / their categories) and merges round-robin, still K=4.

In [3]:
import evaluation_harness as eh
from rag.offer import retrieve, retrieve_multi, build_queries, K
for tc in eh.OFFER_TEST_CASES:
    c = tc.customer_context
    single = [h['id'] for h in retrieve(f"{c['loyalty_tier']} tier benefits, retention offers for at-risk members, free shipping threshold")]
    multi = [h['id'] for h in retrieve_multi(build_queries(c))]
    print(f"{tc.customer_id} {c['loyalty_tier']:8s} single={single}\n{'':23s} multi ={multi}")

/Users/santiago/miniconda3/envs/cs401r-glue/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


CUST-10000317 Gold     single=['POL-LOY-011-4', 'POL-LOY-011-2', 'POL-SHP-007-2', 'POL-LOY-011-5']
                        multi =['POL-LOY-011-4', 'POL-SHP-007-2', 'POL-RET-004-4', 'POL-LOY-011-2']


CUST-10000904 Bronze   single=['POL-LOY-011-4', 'POL-LOY-011-2', 'POL-SHP-007-2', 'KB-FAQ-022-2']
                        multi =['POL-LOY-011-4', 'POL-RET-004-6', 'POL-RET-004-4', 'POL-LOY-011-3']


CUST-10000122 Platinum single=['POL-LOY-011-4', 'POL-LOY-011-2', 'POL-SHP-007-2', 'POL-LOY-011-5']
                        multi =['POL-LOY-011-4', 'POL-RET-004-6', 'POL-RET-004-4', 'POL-LOY-011-2']


CUST-10000588 Silver   single=['POL-LOY-011-4', 'POL-LOY-011-2', 'POL-LOY-011-5', 'POL-SHP-007-2']
                        multi =['POL-LOY-011-4', 'POL-SHP-007-2', 'POL-RET-004-4', 'POL-LOY-011-2']


## 3. Generate an offer for each test customer

`generate_offer` builds the profile text (raw numbers turned into plain meaning), retrieves K=4 chunks, and calls Claude Haiku 4.5 through the `us.` inference profile with the system prompt in `rag/offer.py`.

In [4]:
from rag.offer import generate_offer, RETRIEVAL_MODE
print('retrieval mode:', RETRIEVAL_MODE, '\n')
for tc in eh.OFFER_TEST_CASES:
    r = generate_offer(tc.customer_context, tc.question)
    print(f'== {tc.customer_id} ({tc.customer_context["loyalty_tier"]})')
    print(r['contexts'][0])
    print('chunks:', r['chunk_ids'])
    print('offer :', r['answer'], '\n')

retrieval mode: multi 



== CUST-10000317 (Gold)
Customer profile: Gold member; buys mostly Outdoor and Footwear; shops in a few categories; shops mostly online; hasn't bought in over a month; high churn risk
chunks: ['POL-LOY-011-4', 'POL-SHP-007-2', 'POL-RET-004-4', 'POL-LOY-011-2']
offer : As a Gold member, you currently enjoy free standard shipping on every order and free return shipping on outdoor gear and footwear—making it easier to find your perfect fit without worrying about costs. Earn 1.5× points on your next purchase in these categories and get two free gear repairs per year to keep your equipment in top condition. 



== CUST-10000904 (Bronze)
Customer profile: Bronze member; buys mostly Apparel; shops in one category; shops mostly in store (shipping perks matter less); hasn't bought in over a month; high churn risk
chunks: ['POL-LOY-011-4', 'POL-RET-004-6', 'POL-RET-004-4', 'POL-LOY-011-3']
offer : Come back to Apparel and earn 1 point per $1 spent—your points are valuable and expiring soon. As a Bronze member, you currently earn rewards on every purchase, so your next visit to any of our 400 locations will put you closer to $5 in redemption value. 



== CUST-10000122 (Platinum)
Customer profile: Platinum member; buys mostly Outdoor and Electronics and Home; shops across many categories; shops both online and in store; hasn't bought in over a month; moderate churn risk
chunks: ['POL-LOY-011-4', 'POL-RET-004-6', 'POL-RET-004-4', 'POL-LOY-011-2']
offer : As a Platinum member, you currently earn 2× points on every purchase and enjoy unlimited free gear repair across Outdoor and Electronics categories. Your 60-day return window and free return shipping make it easy to shop with confidence—plus you have access to your $100 annual gear credit through December. Come back and explore what's new across your favorite categories. 



== CUST-10000588 (Silver)
Customer profile: Silver member; buys mostly Home and Beauty; shops in a few categories; shops mostly online; hasn't bought in over a month; high churn risk
chunks: ['POL-LOY-011-4', 'POL-SHP-007-2', 'POL-RET-004-4', 'POL-LOY-011-2']
offer : As a Silver member, you currently earn 1.25× points on every purchase and get free standard shipping on orders over $50—benefits that add up fast on your Home and Beauty favorites. Your next order could be the one that unlocks Gold status, giving you free returns and 24-hour early access to our sales. Shop now and get closer to those Gold-tier rewards. 



## 4. RAGAS evaluation results

Scored with the provided harness (`python -m rag.run_eval`), judge = Claude Haiku 4.5 on Bedrock. Two baseline runs and two multi-query runs. Re-running RAGAS here is skipped to keep the demo cheap; the saved results are below.

In [5]:
import pandas as pd
df = pd.read_csv('docs/lab3-evidence/rag_eval.csv')
m = ['faithfulness', 'answer_relevancy', 'context_recall']
print('judge:', df['judge_model'].unique()[0])
per_run = df.groupby('run')[m].mean().round(3)
per_run

judge: us.anthropic.claude-haiku-4-5-20251001-v1:0


,faithfulness,answer_relevancy,context_recall
run,,,
baseline-run1,0.501,0.362,0.571
baseline-run2,0.622,0.385,0.571
multiquery-run1,0.635,0.350,0.558
multiquery-run2,0.462,0.376,0.558


In [6]:
df.pivot_table(index='customer_id', columns='run', values='context_recall').round(2)

run,baseline-run1,baseline-run2,multiquery-run1,multiquery-run2
customer_id,,,,
CUST-10000122,0.33,0.33,0.33,0.33
CUST-10000317,1.00,1.00,1.00,1.00
CUST-10000588,0.75,0.75,0.50,0.50
CUST-10000904,0.20,0.20,0.40,0.40


Examples, failure cases and fixes: [`docs/lab3-evidence/track-b-examples.md`](../docs/lab3-evidence/track-b-examples.md).